# Multicollinearity Reduction (VIF Elimination)

Removes redundant features from `model_dataset.parquet` using iterative VIF
(Variance Inflation Factor) elimination, before hyperparameter tuning and final
training.

**Why VIF, not pairwise correlation:** pairwise correlation only catches
redundancy between two features at a time. VIF catches multivariate redundancy —
a feature that's well-explained by *several* others combined, even when its
correlation with any single one looks low. This matters here because many
features are variants of the same aggregate across different time windows
(1Y/3Y/6M/3M), which is exactly the kind of multi-feature redundancy pairwise
correlation misses.

**Why VIF is recomputed here rather than reused from `select_vars.ipynb`:**
VIF is only meaningful relative to whichever feature set is currently under
consideration. The VIF values in `numeric_selection_metrics_full_df.xlsx` were
computed per-dataset *before* the tables were joined and filtered down to the
current feature set — they don't reflect the redundancy that exists in the
358-feature joined dataset today.

**Pipeline overview**
1. Load `model_dataset.parquet`
2. Compute VIF (pseudo-inverse method — fast, no per-feature OLS)
3. Iteratively drop the single highest-VIF feature above threshold, recompute, repeat
4. Save the reduced dataset & elimination log

**Output**
- `src/intermediate/model_dataset.parquet` — overwritten in place with reduced features
- `src/Variable-Selection/selected_features.csv` — updated to match
- `src/Variable-Selection/vif_elimination_log.csv` — full audit trail of what was dropped and why

**After this notebook runs:** re-run `02_hyperparameter_tuning.ipynb` and
`03_training_and_analysis.ipynb` — the feature set has changed, so both the search
space results and the final model need to be regenerated against the reduced dataset.

## 1. Imports & Configuration

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")
HPO_DIR  = os.path.join(DATA_ROOT, "HyperParameters", "LightGBM_v1")

os.makedirs(HPO_DIR, exist_ok=True)

PATH_MODEL_DATASET = os.path.join(DATA_ROOT, "ModelDataset", "model_dataset.parquet")
VAR_SELECTION_DIR = os.path.join(DATA_ROOT, "Variable-Selection")
os.makedirs(VAR_SELECTION_DIR, exist_ok=True)

PATH_SELECTED_FEATURES = os.path.join(VAR_SELECTION_DIR, "selected_features.csv")
PATH_ELIMINATION_LOG = os.path.join(VAR_SELECTION_DIR, "vif_elimination_log.csv")
PATH_CORR_MATRIX = os.path.join(VAR_SELECTION_DIR, "correlation_matrix.xlsx")
PATH_VIF_VALUES = os.path.join(VAR_SELECTION_DIR, "vif_values.xlsx")

# Standard textbook threshold. Stricter: 5. Looser: 10 is the most common default
# (e.g. Kutner et al., "Applied Linear Statistical Models").
VIF_THRESHOLD = 10.0

## 2. Load Dataset

VIF only applies to numeric features. Categorical (`category`-dtype) columns,
`SK_ID_CURR`, and `TARGET` are set aside and always kept — they're never candidates
for elimination.

In [2]:
model_dataset = pd.read_parquet(PATH_MODEL_DATASET)

cat_cols = model_dataset.select_dtypes(include=['object', 'category']).columns.tolist()
time_indicator_cols = [c for c in model_dataset.columns  if 'IS_IN_' in c]
numeric_cols = [
    c for c in model_dataset.columns
    if c not in cat_cols + time_indicator_cols + ['SK_ID_CURR', 'TARGET']
]

print(f"Loaded dataset       : {model_dataset.shape}")
print(f"Numeric features (candidates for elimination): {len(numeric_cols)}")
print(f"Categorical features (always kept)           : {len(cat_cols)}")

Loaded dataset       : (307511, 592)
Numeric features (candidates for elimination): 553
Categorical features (always kept)           : 10


## 3. VIF Computation Helper

Same pseudo-inverse approach used in `select_vars_refined.ipynb` — computes VIF
for every feature in a single matrix operation rather than fitting a separate OLS
regression per feature (which is what `statsmodels.variance_inflation_factor` does
and is far too slow to call inside an elimination loop with hundreds of features).

In [3]:
def compute_vif(df, corr_matrix):
    """
    Compute VIF for each feature using pseudo-inverse of the correlation matrix.

    Constant columns (zero variance) are excluded — VIF is undefined for them and
    they contribute no signal anyway.

    Returns
    -------
    pd.DataFrame with columns ['feature', 'VIF'], sorted descending by VIF.
    """
    feature_cols = corr_matrix.columns
    inverse_matrix = np.linalg.pinv(corr_matrix.values)
    vifs = np.diag(inverse_matrix)

    vif_df = pd.DataFrame({'feature': feature_cols, 'VIF': vifs})
    return vif_df.sort_values('VIF', ascending=False).reset_index(drop=True)

## 4. Iterative VIF Elimination

At each step: compute VIF for all remaining features, drop the single worst
offender if it's above `VIF_THRESHOLD`, and repeat. This naturally dissolves
correlated groups one member at a time — no need to define clusters or pick a
"representative" feature up front, since whichever feature remains lowest-VIF
after its correlated siblings are removed survives automatically.

Median imputation is used only for the VIF calculation itself (missing values
would otherwise break the correlation matrix) — it does **not** modify the
underlying dataset.

In [ ]:
from sklearn.impute import SimpleImputer

elimination_log = []
iteration = 0

# Impute once up front for the VIF calculation only — original NaNs are preserved
# in model_dataset itself, this imputed copy is used purely for computing VIF.
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(model_dataset[numeric_cols]),
    columns=numeric_cols,
)

print(f"Starting elimination — {len(numeric_cols)} candidate features, "
      f"threshold VIF > {VIF_THRESHOLD}\n")

# remove constant columns (zero variance) from the VIF calculation, since they have no predictive value and VIF is undefined for them.
constant_cols = [c for c in numeric_cols if X_imputed[c].nunique(dropna=False) <= 1]
remaining_features = [c for c in numeric_cols if c not in constant_cols]
corr_matrix = X_imputed[remaining_features].corr()

Starting elimination — 553 candidate features, threshold VIF > 10.0



In [ ]:
current_corr = corr_matrix.copy()
while True:
    # 2. Slice the existing correlation matrix
    current_corr = current_corr.loc[remaining_features, remaining_features]
    vif_df = compute_vif(X_imputed, current_corr)
    max_vif_row = vif_df.iloc[0]

    if max_vif_row['VIF'] <= VIF_THRESHOLD:
        print(f"Stopped: max VIF = {max_vif_row['VIF']:.2f} "
              f"({max_vif_row['feature']}) is at or below threshold.")
        break

    dropped_feature = max_vif_row['feature']
    dropped_vif = max_vif_row['VIF']

    elimination_log.append({
        'iteration': iteration,
        'dropped_feature': dropped_feature,
        'vif_at_removal': dropped_vif,
        'features_remaining_after': len(remaining_features) - 1,
    })

    remaining_features.remove(dropped_feature)
    X_imputed = X_imputed[remaining_features]
    iteration += 1

    if iteration % 10 == 0:
        print(f"  [{iteration:03d}] dropped '{dropped_feature}' "
              f"(VIF={dropped_vif:.1f}) — {len(remaining_features)} features remain")

print(f"\nElimination complete: {iteration} features dropped, "
      f"{len(remaining_features)} features remain.")

  [010] dropped 'CREDITCARD_AVG_DRAWING_TO_PYMNT_RATIO_PREVLOAN_MEAN' (VIF=13447353307.9) — 613 features remain
  [020] dropped 'CREDITCARD_AVG_ATM_DRAWING_CNT_RATIO_PREVLOAN_MEAN' (VIF=20703865.1) — 603 features remain
  [030] dropped 'CREDITCARD_AMT_TOTAL_RECEIVABLE_MEAN_LAST3Y' (VIF=211201.1) — 593 features remain
  [040] dropped 'CREDITCARD_AMT_BALANCE_MAX_LAST1Y' (VIF=62945.2) — 583 features remain
  [050] dropped 'CREDITCARD_AMT_RECIVABLE_MIN_LAST1Y' (VIF=21322.1) — 573 features remain
  [060] dropped 'CREDITCARD_AMT_BALANCE_MAX_LAST3Y' (VIF=5420.7) — 563 features remain
  [070] dropped 'CREDITCARD_AVG_POS_DRAWING_CNT_RATIO_PREVLOAN_MAX' (VIF=2654.6) — 553 features remain
  [080] dropped 'CREDITCARD_AMT_RECEIVABLE_PRINCIPAL_STD_LAST6M' (VIF=1657.3) — 543 features remain
  [090] dropped 'CREDITCARD_AMT_DRAWINGS_ATM_CURRENT_STD_LAST1Y' (VIF=742.4) — 533 features remain
  [100] dropped 'CREDITCARD_AMT_DRAWINGS_CURRENT_MEAN_LAST6M' (VIF=486.4) — 523 features remain
  [110] dropped 'C

## 5. Diagnostics

In [ ]:
corr_matrix_final = X_imputed[remaining_features].corr()
final_vif = compute_vif(X_imputed, corr_matrix_final)

print(f"Features before elimination : {len(numeric_cols)}")
print(f"Features after elimination  : {len(remaining_features)}")
print(f"Features dropped            : {len(numeric_cols) - len(remaining_features)}")
print(f"Max VIF after elimination   : {final_vif['VIF'].max():.2f}")
print(f"Mean VIF after elimination  : {final_vif['VIF'].mean():.2f}")

elimination_log_df = pd.DataFrame(elimination_log)
if len(elimination_log_df) > 0:
    print("\nTop 15 highest-VIF features dropped:")
    print(
        elimination_log_df
        .sort_values('vif_at_removal', ascending=False)
        .head(15)[['dropped_feature', 'vif_at_removal']]
        .to_string(index=False, float_format='{:.1f}'.format)
    )

Features before elimination : 623
Features after elimination  : 336
Features dropped            : 287
Max VIF after elimination   : 9.67
Mean VIF after elimination  : 3.97

Top 15 highest-VIF features dropped:
                                    dropped_feature  vif_at_removal
         CREDITCARD_AMT_TOTAL_RECEIVABLE_MIN_LAST6M 1237962581916.3
 CREDITCARD_AVG_ATM_DRAWING_AMT_RATIO_PREVLOAN_MEAN  604803256448.4
               CREDITCARD_AMT_RECIVABLE_MEAN_LAST6M   13759447694.8
CREDITCARD_AVG_DRAWING_TO_PYMNT_RATIO_PREVLOAN_MEAN   13447353307.9
           CREDITCARD_AMT_BALANCE_AVG_PREVLOAN_MEAN    4096569643.5
               CREDITCARD_AMT_RECIVABLE_MEAN_LAST3M    3496893427.5
 CREDITCARD_AMT_RECIVABLE_QOQ_PCT_CHANGE_MIN_LAST6M    2642415225.2
 CREDITCARD_AVG_POS_DRAWING_AMT_RATIO_PREVLOAN_MEAN    2132365157.7
    CREDITCARD_AVG_INT_COVERAGE_RATIO_PREVLOAN_MEAN    1118503125.9
         CREDITCARD_AMT_TOTAL_RECEIVABLE_MIN_LAST1Y     815657132.5
 CREDITCARD_AVG_POS_DRAWING_CNT_RATIO_PREV

## 6. Save Reduced Dataset & Audit Trail

`model_dataset.parquet` is overwritten in place with the reduced feature set —
this is a pipeline stage, so downstream notebooks (`hyperparameter_tuning.ipynb`,
`training_and_analysis.ipynb`) don't need to know or care which stage last wrote
this file, only that it contains the current feature set. Categorical columns and
`TARGET` are untouched.

`vif_elimination_log.csv` is the audit trail — which feature was dropped at each
step and its VIF at the moment of removal, useful if you ever need to justify or
revisit why a specific feature was excluded.

In [ ]:
final_feature_cols = remaining_features + cat_cols + time_indicator_cols
reduced_dataset = model_dataset[['SK_ID_CURR'] + final_feature_cols + ['TARGET']]

corr_matrix_final.to_excel(PATH_CORR_MATRIX, index=True)
print(f"Saved correlation matrix → {PATH_CORR_MATRIX}")

final_vif.to_excel(PATH_VIF_VALUES, index=False)
print(f"Saved final VIF values → {PATH_VIF_VALUES}")

reduced_dataset.to_parquet(PATH_MODEL_DATASET, index=False)
print(f"Saved reduced model_dataset.parquet ({reduced_dataset.shape}) → {PATH_MODEL_DATASET}")

pd.Series(final_feature_cols).to_csv(PATH_SELECTED_FEATURES, index=False)
print(f"Saved updated feature list → {PATH_SELECTED_FEATURES}")

elimination_log_df.to_csv(PATH_ELIMINATION_LOG, index=False)
print(f"Saved elimination log ({len(elimination_log_df)} rows) → {PATH_ELIMINATION_LOG}")

Saved reduced model_dataset.parquet ((307511, 360)) → a:\Dominic\My Projects\Home Credit\data\ModelDataset\model_dataset.parquet
Saved updated feature list → a:\Dominic\My Projects\Home Credit\src\Variable-Selection\selected_features.csv
Saved elimination log (287 rows) → a:\Dominic\My Projects\Home Credit\src\Variable-Selection\vif_elimination_log.csv


## 7. Cleanup

In [28]:
del model_dataset, X_imputed, reduced_dataset
import gc; gc.collect()
print("All DataFrames released.")

print("\nNext step: re-run 02_hyperparameter_tuning.ipynb and "
      "03_training_and_analysis.ipynb against the reduced feature set.")

All DataFrames released.

Next step: re-run 02_hyperparameter_tuning.ipynb and 03_training_and_analysis.ipynb against the reduced feature set.
